# Intervention example
In this notebook, let's think about how we might consider and cost an intervention against an infectious disease.
We'll consider one illustrative example, being a new vaccine that has
recently been invented to combat disease X.

We'll build the model with a simlar approach to what you've already seen,
but add a vaccinated compartment that is partially protected against infection.
(The vaccine is "leaky" in the sense that the rate at which the vaccinated are infected is reduced,
but the vaccinated can still be infected.)
This will give us some estimate of the effect of the vaccine in reducing 
the key modelled quantity we are trying to influence, in this case hospitalisations,
and use this to quantify the cost-effectiveness.
![](../images/sir_vacc.svg)

In [ ]:
%pip uninstall orbax-checkpoint flax dopamine-rl --yes
%pip install summerepi2==1.3.6
import pandas as pd
from plotly import graph_objects as go
pd.options.plotting.backend = "plotly"
import logging
logging.getLogger("jax").setLevel(logging.CRITICAL)

from summer2 import CompartmentalModel
from summer2.parameters import Parameter

In [ ]:
population = 1e6  
seed = 1.0
start_time = 0.0
end_time = 50.0

# Split the previous susceptible compartment into two: vaccinated and unvaccinated
model_comps = [
    "vaccinated",
    "unvaccinated", 
    "infectious_hosp", 
    "infectious_nonhosp", 
    "recovered",
]
infect_comps = [
    "infectious_hosp", 
    "infectious_nonhosp",
]

# Build the model as previously
model = CompartmentalModel(
    times=[start_time, end_time], 
    compartments=model_comps, 
    infectious_compartments=infect_comps, 
    timestep=0.2,
)

# Split the susceptible population by vaccination status according to the coverage of the program
suscept_pop = population - seed
start_pop = {
    "unvaccinated": suscept_pop * (1.0 - Parameter("vacc_coverage")), 
    "vaccinated": suscept_pop * Parameter("vacc_coverage"),
    "infectious_nonhosp": seed,
}
model.set_initial_population(start_pop)

# Loop over the infection code for infection for both vaccination statuses
for vacc_group in ["vaccinated", "unvaccinated"]:
    force_of_infection = Parameter("contact_rate")

    # For the vaccinated group, reduce the force of infection by the vaccination effect
    if vacc_group == "vaccinated":
        force_of_infection *= (1.0 - Parameter("vacc_effect"))
        
    admission_rate = Parameter("hosp_fraction") * force_of_infection
    non_admission_rate = (1.0 - Parameter("hosp_fraction")) * force_of_infection
    model.add_infection_frequency_flow(
        name="infection_hosp", 
        contact_rate=admission_rate, 
        source=vacc_group, 
        dest="infectious_hosp",
    )
    model.add_infection_frequency_flow(
        name="infection_nonhosp", 
        contact_rate=non_admission_rate, 
        source=vacc_group, 
        dest="infectious_nonhosp",
    )

model.add_transition_flow(
    name="recovery_hosp", 
    fractional_rate=Parameter("recovery_rate"), 
    source="infectious_hosp", 
    dest="recovered",
)
model.add_transition_flow(
    name="recovery_nonhosp", 
    fractional_rate=Parameter("recovery_rate"), 
    source="infectious_nonhosp", 
    dest="recovered",
)
model.request_output_for_flow("infection_hosp", "infection_hosp");

In [ ]:
parameters = {
    "contact_rate": 1.5,
    "recovery_rate": 0.2,
    "hosp_fraction": 0.1,
    "vacc_effect": 0.8,
    "vacc_coverage": 0.0,
}
model.run(parameters)
base_states = model.get_outputs_df()
int_params = {"vacc_coverage": 0.5}
model.run(parameters | int_params)
int_states = model.get_outputs_df()
hosp_states = pd.DataFrame(
    {
        "base": base_states["infectious_hosp"],
        "int": int_states["infectious_hosp"],
    }
)
cum_hosp_states = hosp_states.cumsum()
cum_hosp_states.plot()

## Cost effectiveness calculations
Calculate the total costs of the program (in some unspecified unit of currency).

In [ ]:
n_vaccinated = suscept_pop * int_params["vacc_coverage"]
cost_per_vacc = 50.0
program_cost = n_vaccinated * cost_per_vacc
print(f"Program cost is {round(program_cost / 1e6, 1)} million.")

Calculate the number of hospitalisations averted during the epidemic
by subtracting the number of hospitalisations under the intervention scenario
from the number of hospitalisations under the base case simulation.

In [ ]:
cum_averted_hosps = cum_hosp_states["base"].iloc[-1] - cum_hosp_states["int"].iloc[-1]
print(f"{round(cum_averted_hosps)} hospitalisations were averted during the simulation period.")

Divide the program cost by the averted hospitalisations to calculate the cost per hospitalisation averted.

In [ ]:
cost_per_hosp_averted = program_cost / cum_averted_hosps
print(f"Cost per hospitalisation averted is {round(cost_per_hosp_averted)}")